# Social-media-only models -- predicting daily change in Polymarket Trump probability

Three models using Bluesky and Reddit social media features:
1. Linear Regression
2. Random Forest
3. Neural Network (MLP)

**Target:** daily change in `polymarket_trump_prob`  
**Features:** 39 Bluesky + Reddit columns (same-day)  
**Splits/Metrics:** same methodology as `1_lag.ipynb`

<!-- toc -->
## Contents
- [1. Setup](#1-setup)
- [2. Load data & compute target](#2-load-data-compute-target)
- [3. Train/val/test split](#3-trainvaltest-split)
- [4. CV folds](#4-cv-folds)
- [5. Helper functions](#5-helper-functions)
- [6. Model 1 -- Linear Regression](#6-model-1----linear-regression)
- [7. Model 2 -- Random Forest](#7-model-2----random-forest)
- [8. Model 3 -- Neural Network (MLP)](#8-model-3----neural-network-mlp)
- [9. Model comparison](#9-model-comparison)


## 1. Setup

In [17]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

warnings.filterwarnings("ignore")

sys.path.insert(0, "../../../")
from Functions.data_splits import get_cv_folds, get_test_split, print_fold_summary, validate_no_leakage
from Functions.evaluation_metrics import directional_accuracy, compute_metrics, cv_evaluate, final_eval

RANDOM_STATE = 42
TEST_DAYS    = 14
N_SPLITS     = 3
GAP          = 1

SOCIAL_COLS = [
    # ── Bluesky volume & author diversity ─────────────────────────────────────
    "bsky_trump_unique_authors", "bsky_harris_unique_authors",
    "bsky_trump_share", "bsky_harris_share", "bsky_posts_per_author",
    # ── Bluesky sentiment ─────────────────────────────────────────────────────
    "bsky_harris_sent_avg", "bsky_harris_pos_share", "bsky_harris_neg_share",
    "bsky_election_sent_avg", "bsky_election_sent_std",
    "bsky_election_pos_share", "bsky_election_neg_share",
    "bsky_sentiment_gap", "bsky_sentiment_gap_lag1",
    # ── Bluesky NRC emotions ──────────────────────────────────────────────────
    "bsky_trump_fear", "bsky_trump_trust", "bsky_trump_surprise",
    "bsky_trump_sadness", "bsky_trump_disgust", "bsky_trump_joy",
    "bsky_harris_fear", "bsky_harris_anger", "bsky_harris_anticipation",
    "bsky_harris_trust", "bsky_harris_disgust", "bsky_harris_joy",
    "bsky_election_surprise", "bsky_election_joy",
    # ── Reddit volume & engagement ────────────────────────────────────────────
    "reddit_unique_authors", "reddit_avg_score",
    "reddit_avg_upvote_ratio", "reddit_avg_comments",
    "reddit_harris_unique_authors", "reddit_harris_share",
    "reddit_conservative_share", "reddit_posts_per_author",
    # ── Reddit sentiment ──────────────────────────────────────────────────────
    "reddit_trump_sent_avg", "reddit_trump_sent_std",
    "reddit_harris_sent_std",
    "reddit_sentiment_gap", "reddit_sentiment_gap_lag1",
    # ── Reddit NRC emotions ───────────────────────────────────────────────────
    "reddit_trump_fear", "reddit_trump_anger", "reddit_trump_anticipation",
    "reddit_trump_trust", "reddit_trump_surprise", "reddit_trump_sadness",
    "reddit_trump_disgust", "reddit_trump_joy",
    "reddit_harris_anticipation", "reddit_harris_trust",
    "reddit_harris_sadness", "reddit_harris_disgust", "reddit_harris_joy",
    "reddit_election_fear", "reddit_election_anger", "reddit_election_anticipation",
    "reddit_election_surprise", "reddit_election_sadness", "reddit_election_joy",
    "reddit_election_sent_std",
]

## 2. Load data & compute target

In [18]:
df_raw = pd.read_csv("../../../Data/3_Gold/basetable_preprocessed.csv", parse_dates=["date"])
df_raw = df_raw.sort_values("date").reset_index(drop=True)

df = df_raw[["date", "polymarket_trump_prob"] + SOCIAL_COLS].copy()
df["target"] = df["polymarket_trump_prob"].diff()
df = df.dropna().reset_index(drop=True)

print(f"Features: {len(SOCIAL_COLS)}")
print(f"Clean rows: {len(df)}  ({df['date'].min().date()} to {df['date'].max().date()})")
print("\nTarget stats:")
print(df["target"].describe().round(4))

Features: 61
Clean rows: 122  (2024-07-06 to 2024-11-04)

Target stats:
count    122.0000
mean      -0.0004
std        0.0178
min       -0.0600
25%       -0.0100
50%       -0.0003
75%        0.0090
max        0.0800
Name: target, dtype: float64


## 3. Train/val/test split

In [19]:
tv_idx, test_idx = get_test_split(df, test_days=TEST_DAYS)

df_tv   = df.iloc[tv_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

X_tv   = df_tv[SOCIAL_COLS].values
y_tv   = df_tv["target"].values
X_test = df_test[SOCIAL_COLS].values
y_test = df_test["target"].values

print(f"Train/val : {len(df_tv):>4} rows  ({df_tv['date'].min().date()} to {df_tv['date'].max().date()})")
print(f"Test      : {len(df_test):>4} rows  ({df_test['date'].min().date()} to {df_test['date'].max().date()})")

Train/val :  108 rows  (2024-07-06 to 2024-10-21)
Test      :   14 rows  (2024-10-22 to 2024-11-04)


## 4. CV folds

In [20]:
folds = get_cv_folds(df_tv, n_splits=N_SPLITS, gap=GAP, test_days=None)
print_fold_summary(df_tv, folds)

print("\nLeakage validation:")
for i, (tr, va) in enumerate(folds, 1):
    validate_no_leakage(tr, va, df_tv, gap=GAP)
    print(f"  Fold {i}: OK")

Fold   Train start     Train end   #Train     Val start       Val end    #Val
-----------------------------------------------------------------------------
   1    2024-07-06    2024-07-31       26    2024-08-02    2024-08-28      27
   2    2024-07-06    2024-08-27       53    2024-08-29    2024-09-24      27
   3    2024-07-06    2024-09-23       80    2024-09-25    2024-10-21      27

Leakage validation:
  Fold 1: OK
  Fold 2: OK
  Fold 3: OK


## 5. Helper functions

In [21]:
# Helper functions are shared across all model notebooks
from Functions.evaluation_metrics import directional_accuracy, compute_metrics, cv_evaluate, final_eval

## 6. Model 1 -- Ridge Regression

In [22]:
# ── Grid search: Ridge alpha ──────────────────────────────────────────────────
pipe_ridge = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())])
param_grid_ridge = {"ridge__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 500.0]}

gs_ridge = GridSearchCV(
    pipe_ridge, param_grid_ridge, cv=folds,
    scoring="neg_mean_absolute_error", refit=True, n_jobs=-1
)
gs_ridge.fit(X_tv, y_tv)

best_alpha = gs_ridge.best_params_["ridge__alpha"]
print(f"Best alpha : {best_alpha}")
print(f"Best CV MAE: {-gs_ridge.best_score_:.4f}")

Best alpha : 500.0
Best CV MAE: 0.0105


In [23]:
# ── CV evaluation with best params ───────────────────────────────────────────
ridge_factory = lambda: Ridge(alpha=best_alpha)
print("=== Ridge Regression -- CV ===")
lr_cv = cv_evaluate(ridge_factory, folds, X_tv, y_tv, scale=True)
lr_cv.round(4)

=== Ridge Regression -- CV ===
  Fold 1: MAE=0.0117  RMSE=0.0153  DA=0.593  R2=-0.0320
  Fold 2: MAE=0.0081  RMSE=0.0102  DA=0.407  R2=-0.0749
  Fold 3: MAE=0.0117  RMSE=0.0160  DA=0.407  R2=-0.2499
  -- Mean --  MAE=0.0105  RMSE=0.0138  DA=0.469  R2=-0.1189


,MAE,RMSE,Dir. Accuracy,R2
1,0.0117,0.0153,0.5926,-0.0320
2,0.0081,0.0102,0.4074,-0.0749
3,0.0117,0.0160,0.4074,-0.2499
Mean,0.0105,0.0138,0.4691,-0.1189
Std,0.0021,0.0031,0.1069,0.1154


In [24]:
# ── Final evaluation on test set ─────────────────────────────────────────────
lr_model, lr_pred, lr_test = final_eval(ridge_factory, X_tv, y_tv, X_test, y_test, scale=True)
print("Ridge Regression -- Test set:")
for k, v in lr_test.items():
    print(f"  {k}: {v:.4f}")

Ridge Regression -- Test set:
  MAE: 0.0155
  RMSE: 0.0211
  Dir. Accuracy: 0.5714
  R2: -0.0944


## 7. Model 2 -- Random Forest

In [25]:
# ── Grid search: RF hyperparameters ──────────────────────────────────────────
param_grid_rf = {
    "n_estimators"  : [100, 200, 300],
    "max_depth"     : [3, 4, 6, None],
    "min_samples_leaf": [2, 3, 5],
}

gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_rf, cv=folds,
    scoring="neg_mean_absolute_error", refit=True, n_jobs=-1
)
gs_rf.fit(X_tv, y_tv)

print(f"Best params: {gs_rf.best_params_}")
print(f"Best CV MAE: {-gs_rf.best_score_:.4f}")

Best params: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
Best CV MAE: 0.0105


In [26]:
# ── CV evaluation with best params ───────────────────────────────────────────
rf_factory = lambda: RandomForestRegressor(
    **gs_rf.best_params_, random_state=RANDOM_STATE, n_jobs=-1
)
print("=== Random Forest -- CV ===")
rf_cv = cv_evaluate(rf_factory, folds, X_tv, y_tv, scale=False)
rf_cv.round(4)

=== Random Forest -- CV ===
  Fold 1: MAE=0.0114  RMSE=0.0151  DA=0.556  R2=-0.0097
  Fold 2: MAE=0.0085  RMSE=0.0104  DA=0.519  R2=-0.1152
  Fold 3: MAE=0.0116  RMSE=0.0158  DA=0.444  R2=-0.2191
  -- Mean --  MAE=0.0105  RMSE=0.0138  DA=0.506  R2=-0.1147


,MAE,RMSE,Dir. Accuracy,R2
1,0.0114,0.0151,0.5556,-0.0097
2,0.0085,0.0104,0.5185,-0.1152
3,0.0116,0.0158,0.4444,-0.2191
Mean,0.0105,0.0138,0.5062,-0.1147
Std,0.0017,0.0029,0.0566,0.1047


In [ ]:
# ── Final evaluation on test set ─────────────────────────────────────────────
rf_model, rf_pred, rf_test = final_eval(rf_factory, X_tv, y_tv, X_test, y_test, scale=False)
print("Random Forest -- Test set:")
for k, v in rf_test.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# ── Feature importances ──────────────────────────────────────────────────────
fi = pd.Series(rf_model.feature_importances_, index=SOCIAL_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
fi.head(20).plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Random Forest -- top-20 feature importances (social media)")
ax.set_ylabel("Importance")
ax.tick_params(axis="x", labelrotation=45)
plt.tight_layout()
plt.show()

## 8. Model 3 -- Support Vector Machine (SVM)

In [ ]:
# ── Grid search: SVM kernel + C + epsilon (+ gamma for rbf/poly) ─────────────
pipe_svm = Pipeline([("scaler", StandardScaler()), ("svm", SVR())])

param_grid_svm = [
    {
        "svm__kernel" : ["linear"],
        "svm__C"      : [0.01, 0.1, 1.0, 10.0],
        "svm__epsilon": [0.001, 0.01, 0.1],
    },
    {
        "svm__kernel" : ["rbf", "poly"],
        "svm__C"      : [0.01, 0.1, 1.0, 10.0],
        "svm__epsilon": [0.001, 0.01, 0.1],
        "svm__gamma"  : ["scale", "auto"],
    },
]

gs_svm = GridSearchCV(
    pipe_svm, param_grid_svm, cv=folds,
    scoring="neg_mean_absolute_error", refit=True, n_jobs=-1
)
gs_svm.fit(X_tv, y_tv)

print(f"Best params: {gs_svm.best_params_}")
print(f"Best CV MAE: {-gs_svm.best_score_:.4f}")

In [ ]:
# ── CV evaluation with best params ───────────────────────────────────────────
svm_params = {k.replace("svm__", ""): v for k, v in gs_svm.best_params_.items()}
svm_factory = lambda: SVR(**svm_params)
print("=== SVM -- CV ===")
svm_cv = cv_evaluate(svm_factory, folds, X_tv, y_tv, scale=True)
svm_cv.round(4)

In [ ]:
# ── Final evaluation on test set ─────────────────────────────────────────────
svm_model, svm_pred, svm_test = final_eval(svm_factory, X_tv, y_tv, X_test, y_test, scale=True)
print("SVM -- Test set:")
for k, v in svm_test.items():
    print(f"  {k}: {v:.4f}")

## 9. Model comparison

In [ ]:
cv_summary = pd.DataFrame({
    "Ridge Regression": lr_cv.loc["Mean"],
    "Random Forest"   : rf_cv.loc["Mean"],
    "SVM"             : svm_cv.loc["Mean"],
}).T.round(4)
print("CV performance (mean across folds):")
display(cv_summary)

In [ ]:
test_summary = pd.DataFrame({
    "Ridge Regression": lr_test,
    "Random Forest"   : rf_test,
    "SVM"             : svm_test,
}).T.round(4)
print("Test set performance:")
display(test_summary)

In [ ]:
test_dates = df_test["date"].values
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, (label, pred, col) in zip(axes, [
    ("Ridge Regression", lr_pred,  colors[0]),
    ("Random Forest",    rf_pred,  colors[1]),
    ("SVM",              svm_pred, colors[2]),
]):
    ax.plot(test_dates, y_test, label="Actual",    color="black", linewidth=1.5)
    ax.plot(test_dates, pred,   label="Predicted", color=col,     linewidth=1.5, linestyle="--")
    ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.set_title(label, fontweight="bold")
    ax.set_ylabel("delta Trump prob")
    ax.legend(loc="upper right", fontsize=9)
axes[-1].set_xlabel("Date")
fig.suptitle("Test-set predictions vs actuals\nDaily delta Polymarket Trump probability -- Social media features only", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
metrics = ["MAE", "RMSE", "Dir. Accuracy", "R2"]
models  = ["Ridge Regression", "Random Forest", "SVM"]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, metric in zip(axes, metrics):
    vals = [test_summary.loc[m, metric] for m in models]
    bars = ax.bar(models, vals, color=colors, edgecolor="white")
    ax.set_title(metric, fontweight="bold")
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=20, ha="right", fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Test-set metric comparison -- Social media-only models", fontsize=12)
plt.tight_layout()
plt.show()